# Exercise 9: Zambian Hydro Model

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>


In [ ]:
lesson_number = 9
print(f'lesson{lesson_number}')

## Colab preparation

In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = False #@param {type:"boolean"}

import os

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = False #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install git+https://github.com/pypsa/pypsa
  !pip install pypsa[excel] 
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

In [ ]:
#@title Download the file for this notebook {display-mode:"form"}
DOWNLOAD_FILE = False #@param {type:"boolean"}

import urllib.request
import os

if DOWNLOAD_FILE:
    url = "https://raw.githubusercontent.com/PriyeshGosai/ich_course_2026/46e109dd68c635d83046eec03d2d2a4afc669366/n_04_zm_hydro_model.xlsx"
    file_name = url.split('/')[-1]  # Extract filename from URL
    filename = file_name
    urllib.request.urlretrieve(url, filename)
    print(f"File downloaded successfully: {os.path.abspath(filename)}")

else:
    print("Skipping file download.")

## Function

In [ ]:
def setup_gurobi_license_file():
    from google.colab import userdata
    import os
    
    wls_access_id = userdata.get('WLSACCESSID')
    wls_secret = userdata.get('WLSSECRET')
    license_id = userdata.get('LICENSEID')
    
    license_content = f"""WLSACCESSID={wls_access_id}
                        WLSSECRET={wls_secret}
                        LICENSEID={license_id}
                        """
    
    os.makedirs('/opt/gurobi', exist_ok=True)
    with open('/opt/gurobi/gurobi.lic', 'w') as f:
        f.write(license_content)
    
    print("✓ License file created")
    print(f"  License ID: {license_id}")


## Model  1

In [ ]:
file_name = 'n_04_zm_hydro_model.xlsx'
# setup_gurobi_license_file()

In [ ]:
import pypsa                      
import pandas as pd               
import plotly.graph_objects as go 
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('plotting.backend', 'plotly')
pypsa.options.api.new_components_api = True


n = pypsa.Network(file_name)


n.optimize(include_objective_constant=True, solver_name='gurobi')


In [ ]:
n.generators.dynamic.p.plot()

In [ ]:
n.links.dynamic.p0.plot()

In [ ]:

n.stores.dynamic.e.plot()

In [ ]:
n.generators.dynamic.p.plot()

In [ ]:
n.buses.dynamic.marginal_price.plot()

In [ ]:
n.loads.dynamic.p_set.plot()

In [ ]:
n.links.dynamic.p0.plot()

In [ ]:
n.processes.dynamic.p.plot()

In [ ]:
n.processes.dynamic.p0.plot()

In [ ]:
n.processes.dynamic.p1.plot()

In [ ]:
(n.processes.dynamic.p2*-1).plot()

In [ ]:
n.stores.dynamic.e.plot()

## Stochastic Model

In [ ]:
import pypsa                      
import pandas as pd               
import plotly.graph_objects as go 
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('plotting.backend', 'plotly')
pypsa.options.api.new_components_api = True


n_scenario = pypsa.Network(file_name)

hydro_generators = {
    'Katima Mulilo': 'Zambezi Inflow',
    'Kasaka': 'Kafue Inflow'
}

# Load network and input data
n_scenario = pypsa.Network(file_name)
df_flows = pd.read_excel(file_name, sheet_name='River Inflows-hourly', header=[0, 1])

print("Original generators:")
print(n.generators.static[['carrier', 'bus']])
print("\ndf_flows shape:", df_flows.shape)
print("\ndf_flows columns:")
print(df_flows.columns.tolist())
print("\ndf_flows head:")
print(df_flows.head())

# ==============================================================================
# SCENARIO SETUP: Using actual inflow scenarios from data
# ==============================================================================
# Data has: Very High, High, Medium, Low, Very Low columns per station
# Map to 3 scenarios: High, Medium, Low

# Mapping: Excel column (station) -> Generator name
column_to_gen = {
    'Katima Mulilo': 'Zambezi Inflow',
    'Kasaka': 'Kafue Inflow'
}

# Scenario mapping: use High, Medium, Low from the data
scenario_columns = {
    'high': 'High',
    'medium': 'Medium',
    'low': 'Low'
}

scenarios = {
    'high': {
        'prob': 0.33,
        'description': 'High inflows',
        'column': 'High'
    },
    'medium': {
        'prob': 0.34,
        'description': 'Median inflows',
        'column': 'Medium'
    },
    'low': {
        'prob': 0.33,
        'description': 'Low inflows',
        'column': 'Low'
    }
}

# Set scenarios in the network
n_scenario.set_scenarios({s: info['prob'] for s, info in scenarios.items()})

n_scenario.optimize(include_objective_constant=True, solver_name='gurobi')



In [ ]:
n_scenario.generators.dynamic.p

In [ ]:
n_scenario.generators.dynamic.p['high'].plot()